In [1]:
# Parameters
RANDOM_STATE = 42
OUT_DIR = "runs"
RUN_NAME = "ethereum"

In [2]:
# ==========================================
# 0. Import thư viện
# ==========================================
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.base import clone
from scipy.stats import randint, uniform

import warnings
warnings.filterwarnings("ignore")

from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, classification_report, confusion_matrix
)


In [3]:
# ==========================================
# 1. Đọc dữ liệu – giữ giống fraud-detection cũ
# ==========================================

# Đường dẫn cũ trong notebook: 'C:\\uit\\elliptic\\transaction_dataset.csv'
# Bạn chỉnh lại cho phù hợp với máy của bạn.
DATA_PATH = r"D:\elliptic\Ethereum-Fraud-Detection-Dataset\transaction_dataset.csv"

df = pd.read_csv(DATA_PATH, index_col=0)
print("Raw shape:", df.shape)
display(df.head())

# Bỏ 2 cột đầu (Index, Address)
df = df.iloc[:, 2:]
print("Sau khi bỏ 2 cột đầu:", df.shape)


Raw shape: (9841, 50)


,Index,Address,FLAG,Avg min between sent tnx,Avg min between received tnx,Time Diff between first and last (Mins),Sent tnx,Received Tnx,Number of Created Contracts,Unique Received From Addresses,...,ERC20 min val sent,ERC20 max val sent,ERC20 avg val sent,ERC20 min val sent contract,ERC20 max val sent contract,ERC20 avg val sent contract,ERC20 uniq sent token name,ERC20 uniq rec token name,ERC20 most sent token type,ERC20_most_rec_token_type
0,1,0x00009277775ac7d0d59eaad8fee3d10ac6c805e8,0,844.26,1093.71,704785.63,721,89,0,40,...,0.000000,1.683100e+07,271779.920000,0.0,0.0,0.0,39.0,57.0,Cofoundit,Numeraire
1,2,0x0002b44ddb1476db43c868bd494422ee4c136fed,0,12709.07,2958.44,1218216.73,94,8,0,5,...,2.260809,2.260809e+00,2.260809,0.0,0.0,0.0,1.0,7.0,Livepeer Token,Livepeer Token
2,3,0x0002bda54cb772d040f779e88eb453cac0daa244,0,246194.54,2434.02,516729.30,2,10,0,10,...,0.000000,0.000000e+00,0.000000,0.0,0.0,0.0,0.0,8.0,NaN,XENON
3,4,0x00038e6ba2fd5c09aedb96697c8d7b8fa6632e5e,0,10219.60,15785.09,397555.90,25,9,0,7,...,100.000000,9.029231e+03,3804.076893,0.0,0.0,0.0,1.0,11.0,Raiden,XENON
4,5,0x00062d1dd1afb6fb02540ddad9cdebfe568e0d89,0,36.61,10707.77,382472.42,4598,20,1,7,...,0.000000,4.500000e+04,13726.659220,0.0,0.0,0.0,6.0,27.0,StatusNetwork,EOS


Sau khi bỏ 2 cột đầu: (9841, 48)


In [4]:
# ==========================================
# 2. Xử lý biến object, missing, cột phương sai 0
#    (bám sát logic notebook fraud-detection cũ)
# ==========================================

# a) Xác định các cột kiểu object -> xem thử
categories = df.select_dtypes('O').columns
print("Các cột object (categorical) ban đầu:")
print(categories.tolist())

# Trong notebook cũ: "# Drop the two categorical features"
# Ở đây ta drop toàn bộ cột object giống ý tưởng đó.
if len(categories) > 0:
    df.drop(columns=categories, inplace=True)
    print("\nĐã drop các cột object. Shape:", df.shape)

# b) Điền missing bằng median cho tất cả cột numeric
df = df.apply(pd.to_numeric, errors="coerce")  # đảm bảo numeric
df.fillna(df.median(), inplace=True)

print("\nSau fillna median – số missing còn lại mỗi cột:")
print(df.isna().sum().sort_values(ascending=False).head())

# c) Bỏ cột phương sai 0
no_var = df.var() == 0
zero_var_cols = df.var()[no_var].index.tolist()
print("\nCác cột có phương sai = 0:")
print(zero_var_cols)

if zero_var_cols:
    df.drop(columns=zero_var_cols, inplace=True)
    print("Shape sau khi drop zero-variance:", df.shape)


Các cột object (categorical) ban đầu:
[' ERC20 most sent token type', ' ERC20_most_rec_token_type']

Đã drop các cột object. Shape: (9841, 46)

Sau fillna median – số missing còn lại mỗi cột:
FLAG                                       0
Avg min between sent tnx                   0
Avg min between received tnx               0
Time Diff between first and last (Mins)    0
Sent tnx                                   0
dtype: int64

Các cột có phương sai = 0:
[' ERC20 avg time between sent tnx', ' ERC20 avg time between rec tnx', ' ERC20 avg time between rec 2 tnx', ' ERC20 avg time between contract tnx', ' ERC20 min val sent contract', ' ERC20 max val sent contract', ' ERC20 avg val sent contract']
Shape sau khi drop zero-variance: (9841, 39)


In [5]:
# ==========================================
# 3. Drop các cột phân bố quá kì quặc (giống cell 'drops')
# ==========================================

# Trong notebook cũ có đoạn:
# drops = ['min value sent to contract', ' ERC20 uniq sent addr.1']
# df.drop(drops, axis=1, inplace=True)

drop_cols = ['min value sent to contract', ' ERC20 uniq sent addr.1']

existing_drop_cols = [c for c in drop_cols if c in df.columns]
if existing_drop_cols:
    df.drop(columns=existing_drop_cols, inplace=True)
    print("Đã drop cột:", existing_drop_cols)
else:
    print("Không tìm thấy các cột trong drop_cols, bỏ qua bước này.")

print("Shape cuối cùng của df:", df.shape)
display(df.head())


Đã drop cột: ['min value sent to contract', ' ERC20 uniq sent addr.1']
Shape cuối cùng của df: (9841, 37)


,FLAG,Avg min between sent tnx,Avg min between received tnx,Time Diff between first and last (Mins),Sent tnx,Received Tnx,Number of Created Contracts,Unique Received From Addresses,Unique Sent To Addresses,min value received,...,ERC20 uniq rec addr,ERC20 uniq rec contract addr,ERC20 min val rec,ERC20 max val rec,ERC20 avg val rec,ERC20 min val sent,ERC20 max val sent,ERC20 avg val sent,ERC20 uniq sent token name,ERC20 uniq rec token name
0,0,844.26,1093.71,704785.63,721,89,0,40,118,0.000000,...,54.0,58.0,0.0,1.500000e+07,265586.147600,0.000000,1.683100e+07,271779.920000,39.0,57.0
1,0,12709.07,2958.44,1218216.73,94,8,0,5,14,0.000000,...,5.0,7.0,0.0,3.650000e+02,57.632615,2.260809,2.260809e+00,2.260809,1.0,7.0
2,0,246194.54,2434.02,516729.30,2,10,0,10,2,0.113119,...,7.0,8.0,0.0,4.428198e+02,65.189009,0.000000,0.000000e+00,0.000000,0.0,8.0
3,0,10219.60,15785.09,397555.90,25,9,0,7,13,0.000000,...,11.0,11.0,0.0,1.141223e+04,1555.550174,100.000000,9.029231e+03,3804.076893,1.0,11.0
4,0,36.61,10707.77,382472.42,4598,20,1,7,19,0.000000,...,23.0,27.0,0.0,9.000000e+04,4934.232147,0.000000,4.500000e+04,13726.659220,6.0,27.0


In [6]:
df.columns

Index(['FLAG', 'Avg min between sent tnx', 'Avg min between received tnx',
       'Time Diff between first and last (Mins)', 'Sent tnx', 'Received Tnx',
       'Number of Created Contracts', 'Unique Received From Addresses',
       'Unique Sent To Addresses', 'min value received', 'max value received ',
       'avg val received', 'min val sent', 'max val sent', 'avg val sent',
       'max val sent to contract', 'avg value sent to contract',
       'total transactions (including tnx to create contract',
       'total Ether sent', 'total ether received',
       'total ether sent contracts', 'total ether balance',
       ' Total ERC20 tnxs', ' ERC20 total Ether received',
       ' ERC20 total ether sent', ' ERC20 total Ether sent contract',
       ' ERC20 uniq sent addr', ' ERC20 uniq rec addr',
       ' ERC20 uniq rec contract addr', ' ERC20 min val rec',
       ' ERC20 max val rec', ' ERC20 avg val rec', ' ERC20 min val sent',
       ' ERC20 max val sent', ' ERC20 avg val sent',
       

In [7]:
df['total transactions'] =  df ['total transactions (including tnx to create contract']
df['Time Diff (last-first)'] =  df ['Time Diff between first and last (Mins)']
df['UniqueRecvAddr'] =  df ['Unique Received From Addresses']
df=df.drop(columns=['total transactions (including tnx to create contract', 'Time Diff between first and last (Mins)','Unique Received From Addresses'])

In [8]:
df.columns

Index(['FLAG', 'Avg min between sent tnx', 'Avg min between received tnx',
       'Sent tnx', 'Received Tnx', 'Number of Created Contracts',
       'Unique Sent To Addresses', 'min value received', 'max value received ',
       'avg val received', 'min val sent', 'max val sent', 'avg val sent',
       'max val sent to contract', 'avg value sent to contract',
       'total Ether sent', 'total ether received',
       'total ether sent contracts', 'total ether balance',
       ' Total ERC20 tnxs', ' ERC20 total Ether received',
       ' ERC20 total ether sent', ' ERC20 total Ether sent contract',
       ' ERC20 uniq sent addr', ' ERC20 uniq rec addr',
       ' ERC20 uniq rec contract addr', ' ERC20 min val rec',
       ' ERC20 max val rec', ' ERC20 avg val rec', ' ERC20 min val sent',
       ' ERC20 max val sent', ' ERC20 avg val sent',
       ' ERC20 uniq sent token name', ' ERC20 uniq rec token name',
       'total transactions', 'Time Diff (last-first)', 'UniqueRecvAddr'],
      dtype=

In [9]:
# ==========================================
# 4. Tách X, y (giữ logic: y là cột đầu, X là phần còn lại)
# ==========================================

y = df.iloc[:, 0].astype(int)   # nhãn (0/1)
X = df.iloc[:, 1:]
print("X shape:", X.shape, "y shape:", y.shape)

print("\nPhân bố nhãn toàn bộ data:")
print(y.value_counts())


X shape: (9841, 36) y shape: (9841,)

Phân bố nhãn toàn bộ data:
FLAG
0    7662
1    2179
Name: count, dtype: int64


In [10]:
# ==========================================
# 5. Chia train / val / test = 70 / 15 / 15 có stratify (style BLTE)
# ==========================================

TEST_SIZE = 0.15
VAL_SIZE  = 0.15

# Bước 1: tách TEST trước (15%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

# Bước 2: tách TRAIN & VAL từ phần còn lại
val_ratio_in_temp = VAL_SIZE / (1.0 - TEST_SIZE)  # ~0.1765

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=val_ratio_in_temp,
    random_state=RANDOM_STATE,
    stratify=y_temp
)

def show_stats(name, yy):
    yy = np.asarray(yy)
    counts = np.bincount(yy)
    n0 = counts[0] if len(counts) > 0 else 0
    n1 = counts[1] if len(counts) > 1 else 0
    ratio = n1 / (n0 + n1) if (n0 + n1) > 0 else 0
    print(f"{name:5s}: 0 = {n0:7d}, 1 = {n1:5d}, scam_ratio = {ratio:.6f}")

print("Shapes:")
print("  Train:", X_train.shape, y_train.shape)
print("  Val  :", X_val.shape,   y_val.shape)
print("  Test :", X_test.shape,  y_test.shape)

print("\n=== Phân bố nhãn sau khi chia ===")
show_stats("ALL",   y)
show_stats("Train", y_train)
show_stats("Val",   y_val)
show_stats("Test",  y_test)


Shapes:
  Train: (6888, 36) (6888,)
  Val  : (1476, 36) (1476,)
  Test : (1477, 36) (1477,)

=== Phân bố nhãn sau khi chia ===
ALL  : 0 =    7662, 1 =  2179, scam_ratio = 0.221421
Train: 0 =    5363, 1 =  1525, scam_ratio = 0.221400
Val  : 0 =    1149, 1 =   327, scam_ratio = 0.221545
Test : 0 =    1150, 1 =   327, scam_ratio = 0.221395


In [11]:
# ==========================================
# 6. Chuẩn hoá feature (StandardScaler – giống BLTE)
# ==========================================
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

feature_cols = X.columns.tolist()
print("Số lượng feature:", len(feature_cols))


Số lượng feature: 36


In [12]:
# ==========================================
# 7. AutoML đơn giản cho 1 model (Random search trên F1 macro VAL)
#    (style giống BLTE: không k-fold, chỉ train/val)
# ==========================================

def sample_param(dist, rng):
    """
    Lấy 1 giá trị từ distribution:
    - Nếu là scipy.stats (randint, uniform, …) -> dist.rvs
    - Nếu là list/tuple -> chọn random 1 phần tử
    """
    if hasattr(dist, "rvs"):
        return dist.rvs(random_state=rng)
    dist = list(dist)
    return dist[rng.randint(0, len(dist))]

def random_search_single_model(
    name,
    base_estimator,
    param_dist,
    X_train, y_train,
    X_val,   y_val,
    n_iter=30,
    random_state=42,
):
    rng = np.random.RandomState(random_state)
    best_f1_macro = -1.0
    best_params = None

    for i in range(n_iter):
        params = {}
        for k, dist in param_dist.items():
            params[k] = sample_param(dist, rng)

        model = clone(base_estimator)
        model.set_params(**params)
        model.fit(X_train, y_train)

        y_val_pred = model.predict(X_val)

        # chọn F1 macro trên val để đánh giá (giống kiểu BLTE dùng macro)
        f1_macro = f1_score(y_val, y_val_pred, average="macro")

        print(f"[{name}] iter {i+1:02d}/{n_iter:02d} – F1_macro(val) = {f1_macro:.6f}")

        if f1_macro > best_f1_macro:
            best_f1_macro = f1_macro
            best_params = params

    print(f"\n>>> {name} – BEST F1_macro(val) = {best_f1_macro:.6f}")
    print("Best params:", best_params)

    # Train lại trên TRAIN+VAL với best_params trước khi test
    X_train_full = np.vstack([X_train, X_val])
    y_train_full = np.concatenate([y_train, y_val])

    best_model = clone(base_estimator)
    best_model.set_params(**best_params)
    best_model.fit(X_train_full, y_train_full)

    return best_model


In [13]:
# ===== Unsupervised models =====
from sklearn.ensemble import IsolationForest
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import f1_score, precision_recall_curve
from sklearn.base import BaseEstimator, ClassifierMixin

In [14]:
# ==========================================
# Unsupervised helper (score -> threshold -> predict)
# ==========================================

def find_best_threshold_by_f1(y_true, scores, n_grid=200):
    """
    Tìm threshold trên score anomaly (score càng lớn càng bất thường)
    để tối đa F1-macro trên validation.
    """
    scores = np.asarray(scores).reshape(-1)
    y_true = np.asarray(y_true).reshape(-1)

    # tránh scan toàn bộ điểm nếu dữ liệu lớn
    qs = np.linspace(0.01, 0.99, n_grid)
    cand = np.unique(np.quantile(scores, qs))

    best_thr = None
    best_f1 = -1.0

    for thr in cand:
        y_pred = (scores >= thr).astype(int)   # anomaly = 1
        f1m = f1_score(y_true, y_pred, average="macro")
        if f1m > best_f1:
            best_f1 = f1m
            best_thr = float(thr)

    return best_thr, float(best_f1)


def _safe_minmax(x):
    x = np.asarray(x).reshape(-1)
    mn, mx = np.min(x), np.max(x)
    if mx - mn < 1e-12:
        return np.zeros_like(x, dtype=float)
    return (x - mn) / (mx - mn)


class IFDetector(BaseEstimator, ClassifierMixin):
    """
    Wrapper Isolation Forest:
    - fit trên normal class (y=0)
    - score_samples_anomaly: score càng lớn càng bất thường
    - threshold được set sau khi tune bằng validation
    """
    def __init__(
        self,
        n_estimators=200,
        max_samples="auto",
        contamination="auto",
        max_features=1.0,
        bootstrap=False,
        random_state=42,
        n_jobs=-1
    ):
        self.n_estimators = n_estimators
        self.max_samples = max_samples
        self.contamination = contamination
        self.max_features = max_features
        self.bootstrap = bootstrap
        self.random_state = random_state
        self.n_jobs = n_jobs

    def fit(self, X, y=None):
        X = np.asarray(X)
        if y is not None:
            y = np.asarray(y)
            # train unsupervised trên normal (0)
            X_fit = X[y == 0] if np.any(y == 0) else X
        else:
            X_fit = X

        self.model_ = IsolationForest(
            n_estimators=self.n_estimators,
            max_samples=self.max_samples,
            contamination=self.contamination,
            max_features=self.max_features,
            bootstrap=self.bootstrap,
            random_state=self.random_state,
            n_jobs=self.n_jobs,
        )
        self.model_.fit(X_fit)

        # threshold mặc định (sẽ overwrite sau tuning)
        train_scores = self.score_samples_anomaly(X)
        self.threshold_ = float(np.quantile(train_scores, 0.95))
        return self

    def score_samples_anomaly(self, X):
        # sklearn decision_function: normal lớn hơn, anomaly nhỏ hơn
        # đảo dấu để "score lớn = bất thường"
        return -self.model_.decision_function(np.asarray(X))

    def set_threshold(self, thr):
        self.threshold_ = float(thr)
        return self

    def predict(self, X):
        s = self.score_samples_anomaly(X)
        return (s >= self.threshold_).astype(int)

    def predict_proba(self, X):
        s = self.score_samples_anomaly(X)
        p1 = _safe_minmax(s)  # pseudo-proba
        return np.column_stack([1 - p1, p1])


class AutoencoderMLPDetector(BaseEstimator, ClassifierMixin):
    """
    Autoencoder dạng MLPRegressor học tái tạo X -> X.
    Score anomaly = reconstruction error (MSE), càng lớn càng bất thường.
    """
    def __init__(
        self,
        hidden_dim=32,
        activation="relu",
        alpha=1e-4,
        learning_rate_init=1e-3,
        max_iter=100,
        batch_size=256,
        random_state=42
    ):
        self.hidden_dim = hidden_dim
        self.activation = activation
        self.alpha = alpha
        self.learning_rate_init = learning_rate_init
        self.max_iter = max_iter
        self.batch_size = batch_size
        self.random_state = random_state

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=np.float32)
        n_features = X.shape[1]

        if y is not None:
            y = np.asarray(y)
            X_fit = X[y == 0] if np.any(y == 0) else X
        else:
            X_fit = X

        # Kiến trúc đối xứng đơn giản
        h = max(4, int(self.hidden_dim))
        self.model_ = MLPRegressor(
            hidden_layer_sizes=(h, max(4, h // 2), h),
            activation=self.activation,
            alpha=self.alpha,
            learning_rate_init=self.learning_rate_init,
            max_iter=self.max_iter,
            batch_size=self.batch_size,
            early_stopping=False,
            random_state=self.random_state,
        )

        # Autoencoder: target = input
        self.model_.fit(X_fit, X_fit)

        train_scores = self.score_samples_anomaly(X)
        self.threshold_ = float(np.quantile(train_scores, 0.95))
        return self

    def score_samples_anomaly(self, X):
        X = np.asarray(X, dtype=np.float32)
        X_hat = self.model_.predict(X)
        err = np.mean((X - X_hat) ** 2, axis=1)
        return err

    def set_threshold(self, thr):
        self.threshold_ = float(thr)
        return self

    def predict(self, X):
        s = self.score_samples_anomaly(X)
        return (s >= self.threshold_).astype(int)

    def predict_proba(self, X):
        s = self.score_samples_anomaly(X)
        p1 = _safe_minmax(s)  # pseudo-proba
        return np.column_stack([1 - p1, p1])



In [15]:
# ==========================================
# AutoML-style random search cho unsupervised
# - tune hyperparams + threshold trên VAL (F1-macro)
# - train final trên TRAIN+VAL
# - calibrate threshold lại trên TRAIN+VAL labels
# ==========================================

def random_search_unsupervised_model(
    name,
    estimator_cls,
    param_dist,
    X_train, y_train,
    X_val,   y_val,
    n_iter=20,
    random_state=42,
):
    rng = np.random.RandomState(random_state)
    best_f1_macro = -1.0
    best_params = None
    best_thr = None

    for i in range(n_iter):
        params = {k: sample_param(dist, rng) for k, dist in param_dist.items()}
        params["random_state"] = random_state

        model = estimator_cls(**params)
        model.fit(X_train, y_train)

        val_scores = model.score_samples_anomaly(X_val)
        thr, f1m = find_best_threshold_by_f1(y_val, val_scores, n_grid=200)

        print(f"[{name}] iter {i+1:02d}/{n_iter:02d} – F1_macro(val) = {f1m:.6f} | thr={thr:.6f}")

        if f1m > best_f1_macro:
            best_f1_macro = f1m
            best_params = params.copy()
            best_thr = thr

    print(f"\n>>> {name} – BEST F1_macro(val) = {best_f1_macro:.6f}")
    print("Best params:", best_params)
    print("Best threshold (VAL):", best_thr)

    # ===== Train lại trên TRAIN+VAL =====
    X_train_full = np.vstack([X_train, X_val])
    y_train_full = np.concatenate([y_train, y_val])

    best_model = estimator_cls(**best_params)
    best_model.fit(X_train_full, y_train_full)

    # ===== Calibrate threshold lại trên TRAIN+VAL (theo yêu cầu) =====
    full_scores = best_model.score_samples_anomaly(X_train_full)
    final_thr, final_f1 = find_best_threshold_by_f1(y_train_full, full_scores, n_grid=300)
    best_model.set_threshold(final_thr)

    print(f">>> {name} – Recalibrated threshold on TRAIN+VAL = {final_thr:.6f} | F1_macro(train+val) = {final_f1:.6f}")

    return best_model

In [16]:
# ==========================================
# 8.4 Unsupervised AutoML: Isolation Forest
# ==========================================
if_param_dist = {
    "n_estimators": randint(100, 500),
    "max_samples": ["auto", 256, 512, 1024],
    "contamination": ["auto"],      # threshold sẽ tự tune bằng val labels
    "max_features": uniform(0.5, 0.5),  # [0.5, 1.0]
    "bootstrap": [False, True],
}

best_if = random_search_unsupervised_model(
    name="IsolationForest",
    estimator_cls=IFDetector,
    param_dist=if_param_dist,
    X_train=X_train_scaled, y_train=y_train,
    X_val=X_val_scaled,     y_val=y_val,
    n_iter=20,   # tăng lên 30 nếu muốn
    random_state=RANDOM_STATE
)

[IsolationForest] iter 01/20 – F1_macro(val) = 0.450438 | thr=-0.150951
[IsolationForest] iter 02/20 – F1_macro(val) = 0.448252 | thr=-0.075879
[IsolationForest] iter 03/20 – F1_macro(val) = 0.447015 | thr=-0.127419
[IsolationForest] iter 04/20 – F1_macro(val) = 0.450438 | thr=-0.143160
[IsolationForest] iter 05/20 – F1_macro(val) = 0.449052 | thr=-0.077213
[IsolationForest] iter 06/20 – F1_macro(val) = 0.447541 | thr=-0.049005
[IsolationForest] iter 07/20 – F1_macro(val) = 0.452545 | thr=-0.150097
[IsolationForest] iter 08/20 – F1_macro(val) = 0.445408 | thr=-0.077127
[IsolationForest] iter 09/20 – F1_macro(val) = 0.453165 | thr=-0.148107
[IsolationForest] iter 10/20 – F1_macro(val) = 0.446695 | thr=-0.058663
[IsolationForest] iter 11/20 – F1_macro(val) = 0.449093 | thr=-0.152118
[IsolationForest] iter 12/20 – F1_macro(val) = 0.447657 | thr=-0.072751
[IsolationForest] iter 13/20 – F1_macro(val) = 0.445648 | thr=-0.093698
[IsolationForest] iter 14/20 – F1_macro(val) = 0.445648 | thr=-0

In [17]:
# ==========================================
# 8.5 Unsupervised AutoML: Autoencoder (MLP)
# ==========================================
ae_param_dist = {
    "hidden_dim": [16, 32, 64, 128],
    "activation": ["relu", "tanh"],
    "alpha": [1e-5, 1e-4, 1e-3],
    "learning_rate_init": [1e-4, 5e-4, 1e-3, 5e-3],
    "max_iter": [80, 120, 160],
    "batch_size": [128, 256, 512],
}

best_ae = random_search_unsupervised_model(
    name="AutoencoderMLP",
    estimator_cls=AutoencoderMLPDetector,
    param_dist=ae_param_dist,
    X_train=X_train_scaled, y_train=y_train,
    X_val=X_val_scaled,     y_val=y_val,
    n_iter=20,   # tăng lên 30 nếu muốn
    random_state=RANDOM_STATE
)

[AutoencoderMLP] iter 01/20 – F1_macro(val) = 0.471641 | thr=0.000279
[AutoencoderMLP] iter 02/20 – F1_macro(val) = 0.536782 | thr=0.004410
[AutoencoderMLP] iter 03/20 – F1_macro(val) = 0.572133 | thr=0.002259
[AutoencoderMLP] iter 04/20 – F1_macro(val) = 0.457570 | thr=0.195812
[AutoencoderMLP] iter 05/20 – F1_macro(val) = 0.455501 | thr=0.175014
[AutoencoderMLP] iter 06/20 – F1_macro(val) = 0.519309 | thr=0.003897
[AutoencoderMLP] iter 07/20 – F1_macro(val) = 0.455501 | thr=0.078136
[AutoencoderMLP] iter 08/20 – F1_macro(val) = 0.519731 | thr=0.000495
[AutoencoderMLP] iter 09/20 – F1_macro(val) = 0.455501 | thr=0.029299
[AutoencoderMLP] iter 10/20 – F1_macro(val) = 0.459649 | thr=0.025337
[AutoencoderMLP] iter 11/20 – F1_macro(val) = 0.472631 | thr=0.001976
[AutoencoderMLP] iter 12/20 – F1_macro(val) = 0.458957 | thr=0.026135
[AutoencoderMLP] iter 13/20 – F1_macro(val) = 0.472543 | thr=0.000467
[AutoencoderMLP] iter 14/20 – F1_macro(val) = 0.478437 | thr=0.001284
[AutoencoderMLP] ite

In [20]:
# ==========================================
# EVAL ONLY (for unsupervised models)
# - supports model.predict(...)
# - supports model.predict_proba(...) OR model.score_samples_anomaly(...)
# - returns AUC-ROC + per-label precision/recall/F1
# ==========================================

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix
)
import numpy as np
import pandas as pd


def evaluate_unsupervised_detailed(name, model, X_test, y_test, verbose=True):
    y_true = np.asarray(y_test).astype(int)
    y_pred = np.asarray(model.predict(X_test)).astype(int)

    # ----- get score for ROC-AUC (label 1 = anomaly/fraud)
    y_score = None

    # ưu tiên predict_proba nếu wrapper của bạn có
    if hasattr(model, "predict_proba"):
        try:
            proba = model.predict_proba(X_test)
            if proba is not None and len(proba.shape) == 2 and proba.shape[1] >= 2:
                y_score = proba[:, 1]
        except Exception:
            y_score = None

    # fallback: anomaly score (score càng lớn càng bất thường)
    if y_score is None and hasattr(model, "score_samples_anomaly"):
        try:
            y_score = np.asarray(model.score_samples_anomaly(X_test)).reshape(-1)
        except Exception:
            y_score = None

    # ----- overall metrics
    acc = accuracy_score(y_true, y_pred)

    prec_macro = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec_macro  = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1_macro   = f1_score(y_true, y_pred, average="macro", zero_division=0)
    f1_micro   = f1_score(y_true, y_pred, average="micro", zero_division=0)
    prec_weighted = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    rec_weighted  = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1_weighted   = f1_score(y_true, y_pred, average="weighted", zero_division=0)

    # ----- AUC-ROC
    auc_roc = np.nan
    if y_score is not None:
        try:
            auc_roc = roc_auc_score(y_true, y_score)
        except Exception:
            auc_roc = np.nan

    # ----- per-label metrics
    report = classification_report(
        y_true, y_pred,
        labels=[0, 1],
        target_names=["label_0", "label_1"],
        output_dict=True,
        zero_division=0
    )

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    result = {
        "model": name,

        "accuracy": round(float(acc), 4),
        "auc_roc": round(float(auc_roc), 4) if not np.isnan(auc_roc) else np.nan,

        "precision_macro": round(float(prec_macro), 4),
        "recall_macro": round(float(rec_macro), 4),
        "f1_macro": round(float(f1_macro), 4),

        "precision_weighted": round(float(prec_weighted), 4),
        "recall_weighted": round(float(rec_weighted), 4),
        "f1_weighted": round(float(f1_weighted), 4),

        "precision_label_0": round(float(report["label_0"]["precision"]), 4),
        "recall_label_0": round(float(report["label_0"]["recall"]), 4),
        "f1_label_0": round(float(report["label_0"]["f1-score"]), 4),
        "support_label_0": int(report["label_0"]["support"]),

        "precision_label_1": round(float(report["label_1"]["precision"]), 4),
        "recall_label_1": round(float(report["label_1"]["recall"]), 4),
        "f1_label_1": round(float(report["label_1"]["f1-score"]), 4),
        "support_label_1": int(report["label_1"]["support"]),

        "tn": int(cm[0, 0]),
        "fp": int(cm[0, 1]),
        "fn": int(cm[1, 0]),
        "tp": int(cm[1, 1]),
    }

    if verbose:
        print(f"\n===== {name} =====")
        print(f"Accuracy      : {result['accuracy']:.4f}")
        print(f"AUC-ROC       : {result['auc_roc'] if pd.notna(result['auc_roc']) else 'NaN'}")
        print(f"Precision(m)  : {result['precision_macro']:.4f}")
        print(f"Recall(m)     : {result['recall_macro']:.4f}")
        print(f"F1(macro)     : {result['f1_macro']:.4f}")
        print(f"F1(micro)     : {result['f1_macro']:.4f}")
        print(f"F1(weighted)  : {result['f1_weighted']:.4f}")

        print("\nPer-label:")
        print(f"  Label 0 -> P={result['precision_label_0']:.4f}, R={result['recall_label_0']:.4f}, F1={result['f1_label_0']:.4f}, Support={result['support_label_0']}")
        print(f"  Label 1 -> P={result['precision_label_1']:.4f}, R={result['recall_label_1']:.4f}, F1={result['f1_label_1']:.4f}, Support={result['support_label_1']}")

        print("\nConfusion Matrix [[TN, FP], [FN, TP]]:")
        print(cm)

    return result

In [21]:
# ==========================================
# Run eval for your unsupervised models only
# (assuming you already have best_if, best_ae, X_test_scaled, y_test)
# ==========================================
unsup_models = {
    "IF": best_if,
    "AE": best_ae,
}

unsup_results = []
for name, model in unsup_models.items():
    unsup_results.append(evaluate_unsupervised_detailed(name, model, X_test_scaled, y_test, verbose=True))

unsup_results_df = (
    pd.DataFrame(unsup_results)
    .sort_values(by=["f1_label_1", "f1_macro"], ascending=False)
    .reset_index(drop=True)
)

display(unsup_results_df)


===== IF =====
Accuracy      : 0.7380
AUC-ROC       : 0.3355
Precision(m)  : 0.4440
Recall(m)     : 0.4838
F1(macro)     : 0.4463
F1(micro)     : 0.4463
F1(weighted)  : 0.6702

Per-label:
  Label 0 -> P=0.7727, R=0.9400, F1=0.8482, Support=1150
  Label 1 -> P=0.1154, R=0.0275, F1=0.0444, Support=327

Confusion Matrix [[TN, FP], [FN, TP]]:
[[1081   69]
 [ 318    9]]

===== AE =====
Accuracy      : 0.7542
AUC-ROC       : 0.3309
Precision(m)  : 0.4502
Recall(m)     : 0.4909
F1(macro)     : 0.4456
F1(micro)     : 0.4456
F1(weighted)  : 0.6761

Per-label:
  Label 0 -> P=0.7754, R=0.9635, F1=0.8592, Support=1150
  Label 1 -> P=0.1250, R=0.0183, F1=0.0320, Support=327

Confusion Matrix [[TN, FP], [FN, TP]]:
[[1108   42]
 [ 321    6]]


,model,accuracy,auc_roc,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,precision_label_0,...,f1_label_0,support_label_0,precision_label_1,recall_label_1,f1_label_1,support_label_1,tn,fp,fn,tp
0,IF,0.7380,0.3355,0.4440,0.4838,0.4463,0.6272,0.7380,0.6702,0.7727,...,0.8482,1150,0.1154,0.0275,0.0444,327,1081,69,318,9
1,AE,0.7542,0.3309,0.4502,0.4909,0.4456,0.6314,0.7542,0.6761,0.7754,...,0.8592,1150,0.1250,0.0183,0.0320,327,1108,42,321,6
